# Item B

### Carregamento dos dados

In [3]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import BloodMNIST
import os

# Definir a transformação para converter as imagens em Tensores PyTorch [0.0, 1.0]
# Nota: O ToTensor() já divide automaticamente os pixels [0, 255] por 255.0
data_transform = transforms.Compose([
    transforms.ToTensor()
])

NUM_WORKERS = min(4, os.cpu_count())
print(f"NUM_WORKERS: {NUM_WORKERS}")

# Carregar os datasets oficiais do BloodMNIST (Treino, Validação e Teste)
# O parâmetro download=True baixa a base automaticamente caso não exista localmente
train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
val_dataset   = BloodMNIST(split='val', transform=data_transform, download=True)
test_dataset  = BloodMNIST(split='test', transform=data_transform, download=True)

# Criar os DataLoaders para gerenciar o mini-batching
BATCH_SIZE = 64

#train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
#val_loader   = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# gpu?
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"DEVICE: {DEVICE}")


train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == 'cuda' else False,
    persistent_workers=True if NUM_WORKERS > 0 else False  # Mantém os processos vivos entre épocas
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == 'cuda' else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

NUM_WORKERS: 4
DEVICE: cuda


### Treino do Modelo

In [ ]:
import pandas as pd
import ml3.cnn_bc as cnn_bc


num_kernels_list=[8, 16, 32, 64, 128, 256]
kernel_sizes=[3, 5, 7]

results = cnn_bc.fit_cnn(
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_kernels_list=num_kernels_list,
    kernel_sizes=kernel_sizes,
    lr=1e-3,
    epochs=20,
    save_tag="_cnn_item_b"
)

# Converter para DataFrame e visualizar estilo Pandas/Plotly
df_results = pd.DataFrame(results)
df_results.to_csv("item_b_cnn_validation_info.csv", index=False)



--- Testando CNN: num_kernels=8, kernel_size=3 ---

--- Testando CNN: num_kernels=8, kernel_size=5 ---

--- Testando CNN: num_kernels=8, kernel_size=7 ---

--- Testando CNN: num_kernels=16, kernel_size=3 ---

--- Testando CNN: num_kernels=16, kernel_size=5 ---

--- Testando CNN: num_kernels=16, kernel_size=7 ---

--- Testando CNN: num_kernels=32, kernel_size=3 ---

--- Testando CNN: num_kernels=32, kernel_size=5 ---

--- Testando CNN: num_kernels=32, kernel_size=7 ---

--- Testando CNN: num_kernels=64, kernel_size=3 ---

--- Testando CNN: num_kernels=64, kernel_size=5 ---

--- Testando CNN: num_kernels=64, kernel_size=7 ---

--- Testando CNN: num_kernels=128, kernel_size=3 ---

--- Testando CNN: num_kernels=128, kernel_size=5 ---

--- Testando CNN: num_kernels=128, kernel_size=7 ---

--- Testando CNN: num_kernels=256, kernel_size=3 ---

--- Testando CNN: num_kernels=256, kernel_size=5 ---

--- Testando CNN: num_kernels=256, kernel_size=7 ---

Melhor Configuração de CNN Encontrada: {'n

### Plotagem do Treino (Validação)

In [17]:
import pandas as pd

df_results = pd.read_csv("item_b_cnn_validation_info.csv")
df_sorted = df_results.sort_values(by="val_acc", ascending=False).reset_index(drop=True)

df_sorted = df_sorted.rename(columns={
    "num_kernels": "Kernels (K)",
    "kernel_size": "Tamanho (k)"
})

styled_df = df_sorted.style.background_gradient(
    cmap="Blues", subset=["val_acc"]
).format({
    "val_acc": "{:.4f}"
}).set_caption("<b>Progressão da Acurácia Balanceada na Validação da CNN</b>")

# Exibir a tabela no Notebook
styled_df

,Kernels (K),Tamanho (k),val_acc
0,256,3,0.9142
1,256,5,0.9066
2,128,3,0.9056
3,128,5,0.9051
4,64,3,0.8982
5,64,5,0.8957
6,32,5,0.8910
7,32,3,0.8888
8,16,5,0.8847
9,16,3,0.8793


In [23]:
import plotly.express as px

# Pivotear os dados para formato matricial (Linhas = K, Colunas = k)
pivot_df = df_results.pivot(index="num_kernels", columns="kernel_size", values="val_acc")

fig_heatmap = px.imshow(
    pivot_df,
    labels=dict(x="Tamanho do Kernel (k)", y="Quantidade de Kernels (K)", color="Acurácia Val"),
    x=[f"{k}x{k}" for k in pivot_df.columns],
    y=[str(K) for K in pivot_df.index],
    text_auto=".3f",
    #color_continuous_scale="Blues",
    title="<b>Matriz de Acurácia de Validação da CNN (K x k)</b>"
)

fig_heatmap.update_layout(
    template="plotly_white",
    width=600,
    height=450
)

fig_heatmap.show()

# Item C

### Re-treino

In [5]:
import time, torch
import ml3.cnn_bc as cnn_bc
from ml3.config import RANDOM_SEED

#best_cnn = cnn_bc.BloodMNISTCNN(
#   in_channels=3,
#    num_kernels=BEST_NUM_KERNELS,
#    kernel_size=BEST_KERNEL_SIZE,
#    num_classes=8
#)
#best_cnn.load_state_dict(torch.load('best_model_cnn_item_b.pth'))

torch.manual_seed(RANDOM_SEED)          # reprodutibilidade

BEST_NUM_KERNELS = 256
BEST_KERNEL_SIZE = 3

t0 = time.perf_counter()
print("Iniciando treino final!")
best_cnn, val_acc = cnn_bc.wrap_train_and_validate(BEST_NUM_KERNELS, BEST_KERNEL_SIZE, 1e-3, train_loader, val_loader, DEVICE, epochs=20)
train_time = time.perf_counter() - t0
n_params = sum(p.numel() for p in best_cnn.parameters())   # esperado: 408.584
print(f"acc={val_acc}, time={train_time}, n_params={n_params}")

res = cnn_bc.evaluate_model(best_cnn, test_loader, device=DEVICE)

Iniciando treino final!
acc=0.9077533874602884, time=73.1659370999987, n_params=408584


### Teste

In [8]:
import torch
import plotly.express as px
from sklearn.metrics import classification_report
import ml3.cnn_bc as cnn_bc


# Nomes das 8 classes do BloodMNIST
class_names = [
    'Basófilo', 'Eosinófilo', 'Eritroblasto', 'Granulócito Imaturo',
    'Linfócito', 'Monócito', 'Neutrófilo', 'Plaqueta'
]

res = cnn_bc.evaluate_model(best_cnn, test_loader, device=DEVICE)

# Exibir Métricas
print(f"Acurácia Balanceada (Teste): {res['bal_acc']:.4f}")
print(f"F1-Score Macro (Teste)     : {res['f1_macro']:.4f}")

print("--- Relatório Detalhado por Classe ---")
print(classification_report(res['targets'], res['preds'], target_names=class_names, digits=4))

fig_cm = px.imshow(
    res['cm'],
    x=class_names,
    y=class_names,
    text_auto=True,
    color_continuous_scale='Blues',
    title="<b>Matriz de Confusão - CNN (Teste BloodMNIST)</b>",
    labels=dict(x="Classe Predita", y="Classe Real", color="Frequência")
)

fig_cm.update_layout(template="plotly_white", width=700, height=600)
fig_cm.show()

Acurácia Balanceada (Teste): 0.8892
F1-Score Macro (Teste)     : 0.8929
--- Relatório Detalhado por Classe ---
                     precision    recall  f1-score   support

           Basófilo     0.8704    0.8811    0.8758       244
         Eosinófilo     0.9622    0.9792    0.9706       624
       Eritroblasto     0.8913    0.9228    0.9068       311
Granulócito Imaturo     0.7617    0.8446    0.8010       579
          Linfócito     0.8785    0.8930    0.8857       243
           Monócito     0.8857    0.6549    0.7530       284
         Neutrófilo     0.9690    0.9399    0.9543       666
           Plaqueta     0.9936    0.9979    0.9958       470

           accuracy                         0.9062      3421
          macro avg     0.9016    0.8892    0.8929      3421
       weighted avg     0.9086    0.9062    0.9055      3421



### Imagens mal Classificadas

In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

erros = cnn_bc.get_misclassified_samples(best_cnn, test_loader, device=DEVICE, max_samples=5)

titles = [
    f"Real: {class_names[s['true_label']]}<br>Pred: {class_names[s['pred_label']]} ({s['probs'][s['pred_label']]*100:.1f}%)"
    for s in erros
]

fig = make_subplots(
    rows=2, 
    cols=3, 
    subplot_titles=titles,
    vertical_spacing=0.22,
    horizontal_spacing=0.08
)

for i, sample in enumerate(erros):
    # Lógica de empilhamento:
    # i = 0, 1, 2 -> row = 1 | col = 1, 2, 3
    # i = 3, 4    -> row = 2 | col = 1, 2
    row = (i // 3) + 1
    col = (i % 3) + 1
    
    # Converter para uint8 [0, 255]
    img_np = (sample['image'].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    fig.add_trace(go.Image(z=img_np), row=row, col=col)

# 3. Remover marcas dos eixos e ajustar proporções da figura
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.update_layout(
    title="<b>Padrões Classificados Incorretamente (CNN - Item C)</b>",
    height=600,
    width=1000,
    margin=dict(l=20, r=20, t=110, b=20)
)

fig.show()

In [24]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np, pandas as pd

erros = cnn_bc.get_misclassified_samples(best_cnn, test_loader, device=DEVICE, max_samples=5)
n = len(erros)
titles = [f"{i+1}) Real: {class_names[s['true_label']]}<br>Pred: {class_names[s['pred_label']]}"
          for i, s in enumerate(erros)]

fig = make_subplots(rows=2, cols=n, subplot_titles=titles + [""] * n,
                    row_heights=[0.4, 0.6], vertical_spacing=0.1, horizontal_spacing=0.03)
for i, s in enumerate(erros, start=1):
    img = (s['image'].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    fig.add_trace(go.Image(z=img), row=1, col=i)
    cores = ['green' if k == s['true_label'] else 'red' if k == s['pred_label'] else 'lightgray'
             for k in range(8)]
    fig.add_trace(go.Bar(x=s['probs'], y=class_names, orientation='h', marker_color=cores,
                         text=[f"{p:.1%}" for p in s['probs']], textposition='outside'), row=2, col=i)

fig.update_xaxes(showticklabels=False, row=1); fig.update_yaxes(showticklabels=False, row=1)
fig.update_xaxes(range=[0, 1.35], showticklabels=False, row=2)
fig.update_yaxes(autorange='reversed', showticklabels=False, row=2)
fig.update_yaxes(showticklabels=True, row=2, col=1)
fig.update_layout(template="plotly_white", showlegend=False, width=1500, height=600,
                  title="<b>Padrões Classificados Incorretamente (CNN)</b>")
fig.update_layout(margin=dict(t=150))
# Aumentar as fontes dos títulos, das porcentagens e dos rótulos das classes
fig.update_annotations(font_size=19)
fig.update_traces(
    textfont=dict(size=25),
    selector=dict(type="bar")
)
fig.update_yaxes(tickfont=dict(size=16), row=2, col=1)
fig.update_layout(
    title_font_size=26,
    margin=dict(t=170)
)
fig.update_xaxes(range=[0, 1.6], row=2)
fig.show()

df_probs = pd.DataFrame([s['probs'] * 100 for s in erros], columns=class_names,
                        index=[f"{i+1}: {class_names[s['true_label']]}" for i, s in enumerate(erros)]).round(1)
df_probs

,Basófilo,Eosinófilo,Eritroblasto,Granulócito Imaturo,Linfócito,Monócito,Neutrófilo,Plaqueta
1: Linfócito,3.300000,0.0,47.099998,11.500000,37.200001,0.8,0.0,0.0
2: Neutrófilo,0.100000,0.9,96.300003,1.400000,0.900000,0.0,0.0,0.3
3: Neutrófilo,25.299999,1.3,0.100000,20.400000,21.200001,11.8,19.9,0.0
4: Monócito,0.900000,0.0,0.500000,89.300003,0.000000,9.3,0.0,0.0
5: Monócito,4.900000,0.2,4.400000,77.699997,9.900000,2.9,0.1,0.0
